In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from pathlib import Path

# Configuración
BASE_URL = "https://corunacorre.com"
HEADERS = {"User-Agent": "P2-CorunaCorre-BigData-UAC/1.0 (practica academica)"}
PAUSE = 1.5  

#  datos
RAW_DIR = Path("../datos/raw")
INTERIM_DIR = Path("../datos/interim")
EXTERNOS_DIR = Path("../datos/externos")

for d in [RAW_DIR, INTERIM_DIR, EXTERNOS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuración lista")
print(f"   User-Agent: {HEADERS['User-Agent']}")
print(f"   Pausa entre peticiones: {PAUSE}s")

Configuración lista
   User-Agent: P2-CorunaCorre-BigData-UAC/1.0 (practica academica)
   Pausa entre peticiones: 1.5s


In [2]:
def descargar_pagina(url, nombre_archivo):
    """Descarga una página y la guarda en caché en datos/raw/.
    Si ya existe el archivo, lo reutiliza sin volver a descargar.
    (Pasos 8 y 9 de la guía)"""
    ruta = RAW_DIR / nombre_archivo
    
    if ruta.exists():
        print(f"  [CACHÉ] {nombre_archivo}")
        return ruta.read_bytes()
    
    print(f"  [DESCARGA] {url}")
    time.sleep(PAUSE)  # Pausa obligatoria >= 1 segundo
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    ruta.write_bytes(resp.content)
    return resp.content

print("Función de descarga con caché definida")

Función de descarga con caché definida


In [3]:

html_eventos = descargar_pagina(f"{BASE_URL}/gl/web/resultado/edicion-63/eventos", "eventos_edicion_63.html")
soup_eventos = BeautifulSoup(html_eventos, "lxml")

enlaces_eventos = []
for a in soup_eventos.find_all("a", href=True):
    href = a["href"]
    if "/gl/web/evento/" in href:
        if href not in [e["url"] for e in enlaces_eventos]:
            enlaces_eventos.append({
                "url": href,
                "id_evento": href.strip("/").split("/")[-1]
            })

print(f"Total eventos encontrados: {len(enlaces_eventos)}")

carreras_absoluta = []

for ev in enlaces_eventos:
    id_ev = ev["id_evento"]
    nombre_archivo = f"evento_{id_ev}_resultados.html"
    url_res_evento = f"{BASE_URL}/gl/web/resultado/evento-{id_ev}"
    
    html_ev = descargar_pagina(url_res_evento, nombre_archivo)
    soup_ev = BeautifulSoup(html_ev, "lxml")
    
    titulo_tag = soup_ev.find("h1")
    nombre_carrera = titulo_tag.get_text(strip=True) if titulo_tag else f"Carrera {id_ev}"
    nombre_carrera = nombre_carrera.replace("Resultados de", "").strip()
    
    url_absoluta = None
    for a in soup_ev.find_all("a", href=True):
        texto_link = a.get_text(strip=True).upper()
        # Buscamos "ABSOLUTA" y excluimos explícitamente infantiles/juveniles
        if "ABSOLUTA" in texto_link and not any(k in texto_link for k in ["PITUFOS", "BENXAMIN", "ALEVIN", "INFANTIL", "CADETE", "XUVENIL"]):
            url_absoluta = a["href"]
            break
            
    if url_absoluta:
        carreras_absoluta.append({
            "id_evento": id_ev,
            "nombre": nombre_carrera,
            "url_absoluta": url_absoluta
        })
        print(f"  [VÁLIDA] Evento {id_ev} ({nombre_carrera}) -> {url_absoluta}")
    else:
        print(f"  [EXCLUIDA] Evento {id_ev} ({nombre_carrera}) -> No tiene categoría Absoluta (aún no celebrada)")

print(f"\n Total de carreras con resultados de categoría Absoluta: {len(carreras_absoluta)}")

  [DESCARGA] https://corunacorre.com/gl/web/resultado/edicion-63/eventos
Total eventos encontrados: 7
  [DESCARGA] https://corunacorre.com/gl/web/resultado/evento-3210
  [VÁLIDA] Evento 3210 (RESULTADOSACTUALEVENTOSTorre de Hercules) -> /gl/web/resultado/competicion-17059
  [DESCARGA] https://corunacorre.com/gl/web/resultado/evento-3211
  [VÁLIDA] Evento 3211 (RESULTADOSACTUALEVENTOSMatogrande-Xuxán) -> /gl/web/resultado/competicion-17137
  [DESCARGA] https://corunacorre.com/gl/web/resultado/evento-3212
  [VÁLIDA] Evento 3212 (RESULTADOSACTUALEVENTOSSan Pedro de Visma) -> /gl/web/resultado/competicion-17292
  [DESCARGA] https://corunacorre.com/gl/web/resultado/evento-3213
  [VÁLIDA] Evento 3213 (RESULTADOSACTUALEVENTOSOs Rosales) -> /gl/web/resultado/competicion-17393
  [DESCARGA] https://corunacorre.com/gl/web/resultado/evento-3214
  [VÁLIDA] Evento 3214 (RESULTADOSACTUALEVENTOSVolta a Oza) -> /gl/web/resultado/competicion-17546
  [DESCARGA] https://corunacorre.com/gl/web/resultado/ev

In [20]:

def parsear_pagina_resultados(html_bytes, id_carrera, num_pagina):
    """Extrae las filas de resultados de la tabla de una página."""
    soup = BeautifulSoup(html_bytes, "lxml")
    tabla = soup.find("table")
    if not tabla:
        return []
    
    tbody = tabla.find("tbody")
    filas_html = tbody.find_all("tr") if tbody else tabla.find_all("tr")[1:]
    
    filas_datos = []
    for tr in filas_html:
        tds = tr.find_all("td")
        if len(tds) < 8:
            continue
        
        filas_datos.append({
            "id_carrera": id_carrera,
            "pagina_origen": num_pagina,
            "puesto": tds[1].get_text(strip=True),
            "dorsal": tds[2].get_text(strip=True),
            "nombre": tds[3].get_text(strip=True),
            "apellidos": tds[4].get_text(strip=True),
            "nombre_completo": f"{tds[3].get_text(strip=True)} {tds[4].get_text(strip=True)}".strip(),
            "puesto_sexo": tds[5].get_text(strip=True),
            "puesto_categoria": tds[6].get_text(strip=True),
            "tiempo_oficial": tds[7].get_text(strip=True)
        })
        
    return filas_datos


todos_los_resultados = {}
registro_extraccion = []

print("Iniciando extracción COMPLETA de las 6 carreras...\n")

for idx, carrera in enumerate(carreras_absoluta, start=1):
    id_carrera = f"carrera_0{idx}"
    id_ev = carrera["id_evento"]
    url_base_cat = f"{BASE_URL}{carrera['url_absoluta']}"
    
    print(f"=== Procesando {id_carrera}: Evento {id_ev} ===")
    
    filas_carrera = []
    pagina = 1
    dorsales_anteriores = set()
    
    while True:
        url_pag = f"{url_base_cat}?page={pagina}"
        nombre_raw = f"{id_carrera}_pag_{pagina:02d}.html"
        
        try:
            html = descargar_pagina(url_pag, nombre_raw)
            filas = parsear_pagina_resultados(html, id_carrera, pagina)
            
            if not filas:
                print(f"  -> Fin detectado: Página {pagina} vacía.")
                (RAW_DIR / nombre_raw).unlink(missing_ok=True)
                break
                
            dorsales_actuales = {f["dorsal"] for f in filas}
            if dorsales_actuales == dorsales_anteriores:
                print(f"  -> Fin detectado: Página {pagina} repite los datos anteriores.")
                (RAW_DIR / nombre_raw).unlink(missing_ok=True)
                break
                
            dorsales_anteriores = dorsales_actuales
            filas_carrera.extend(filas)
            
            if len(filas) < 25:
                print(f"  -> Fin alcanzado en página {pagina} (última página con {len(filas)} filas).")
                break
                
            pagina += 1
            
        except Exception as e:
            print(f" Error en página {pagina}: {e}")
            break
            
    total_paginas_reales = pagina if (filas and len(filas) < 25) else (pagina - 1)
    df_carrera = pd.DataFrame(filas_carrera)
    todos_los_resultados[id_carrera] = df_carrera
    
    archivo_interim = INTERIM_DIR / f"{id_carrera}_interim.csv"
    df_carrera.to_csv(archivo_interim, index=False)
    
    duplicados = df_carrera.duplicated(subset=["dorsal"]).sum() if not df_carrera.empty else 0
    
    registro_extraccion.append({
        "id_carrera": id_carrera,
        "evento_id": id_ev,
        "paginas_totales": total_paginas_reales,
        "total_participantes": len(df_carrera),
        "duplicados_dorsal": duplicados,
        "estado": "OK" if len(df_carrera) > 0 and duplicados == 0 else "REVISAR"
    })
    
    print(f"  {id_carrera} completa: {len(df_carrera)} corredores en {total_paginas_reales} páginas.\n")

print("Extracción 100% completa de todas las carreras")

🚀 Iniciando extracción COMPLETA de las 6 carreras...

=== Procesando carrera_01: Evento 3210 ===
  [CACHÉ] carrera_01_pag_01.html
  [CACHÉ] carrera_01_pag_02.html
  [CACHÉ] carrera_01_pag_03.html
  [CACHÉ] carrera_01_pag_04.html
  [CACHÉ] carrera_01_pag_05.html
  [CACHÉ] carrera_01_pag_06.html
  [CACHÉ] carrera_01_pag_07.html
  [CACHÉ] carrera_01_pag_08.html
  [CACHÉ] carrera_01_pag_09.html
  [CACHÉ] carrera_01_pag_10.html
  [CACHÉ] carrera_01_pag_11.html
  [CACHÉ] carrera_01_pag_12.html
  [CACHÉ] carrera_01_pag_13.html
  [CACHÉ] carrera_01_pag_14.html
  [CACHÉ] carrera_01_pag_15.html
  [DESCARGA] https://corunacorre.com/gl/web/resultado/competicion-17059?page=16
  [DESCARGA] https://corunacorre.com/gl/web/resultado/competicion-17059?page=17
  [DESCARGA] https://corunacorre.com/gl/web/resultado/competicion-17059?page=18
  [DESCARGA] https://corunacorre.com/gl/web/resultado/competicion-17059?page=19
  [DESCARGA] https://corunacorre.com/gl/web/resultado/competicion-17059?page=20
  [DESCA

In [22]:

url_inscritos_base = f"{BASE_URL}/gl/web/inscripcion/listado/"
print("Descargando lista general de inscritos en el circuito...")

filas_inscritos = []
pag_inscritos = 1
dorsales_inscritos_prev = set()

while True:
    nombre_raw = f"inscritos_pag_{pag_inscritos:02d}.html"
    url_pag = f"{url_inscritos_base}?page={pag_inscritos}"
    
    try:
        html = descargar_pagina(url_pag, nombre_raw)
        soup = BeautifulSoup(html, "lxml")
        
        tabla = soup.find("table")
        if not tabla:
            break
            
        tbody = tabla.find("tbody")
        trs = tbody.find_all("tr") if tbody else tabla.find_all("tr")[1:]
        
        if not trs:
            break
            
        # Comprobar si repite la misma página
        filas_pagina = []
        for tr in trs:
            tds = tr.find_all("td")
            if len(tds) < 5:
                continue
            filas_pagina.append({
                "pagina_origen": pag_inscritos,
                "nombre": tds[0].get_text(strip=True),
                "apellidos": tds[1].get_text(strip=True),
                "nombre_completo": f"{tds[0].get_text(strip=True)} {tds[1].get_text(strip=True)}".strip(),
                "dorsal": tds[2].get_text(strip=True),
                "pagado": tds[3].get_text(strip=True),
                "estado_inscripcion": tds[4].get_text(strip=True)
            })
            
        dorsales_act = {f["dorsal"] for f in filas_pagina}
        if dorsales_act == dorsales_inscritos_prev:
            print(f"  -> Fin detectado en página {pag_inscritos} (datos repetidos).")
            (RAW_DIR / nombre_raw).unlink(missing_ok=True)
            break
            
        dorsales_inscritos_prev = dorsales_act
        filas_inscritos.extend(filas_pagina)
        
        if len(trs) < 25:
            print(f"  -> Fin alcanzado en página {pag_inscritos} con {len(trs)} filas.")
            break
            
        pag_inscritos += 1
        
    except Exception as e:
        print(f"  Error en página {pag_inscritos}: {e}")
        break

df_inscritos = pd.DataFrame(filas_inscritos)
archivo_inscritos = INTERIM_DIR / "inscritos_circuito_interim.csv"
df_inscritos.to_csv(archivo_inscritos, index=False)

print(f"\n Total inscritos extraídos: {len(df_inscritos)} en {pag_inscritos} páginas.")
print(f"Archivo guardado en: {archivo_inscritos}")

🚀 Descargando lista general de inscritos en el circuito...
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=1
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=2
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=3
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=4
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=5
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=6
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=7
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=8
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=9
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=10
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=11
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=12
  [DESCARGA] https://corunacorre.com/gl/web/inscripcion/listado/?page=13
 

In [23]:

df_registro = pd.DataFrame(registro_extraccion)
print("=== REGISTRO DE AUDITORÍA Y COBERTURA (CARRERAS) ===")
display(df_registro)

df_registro.to_csv(INTERIM_DIR / "registro_extraccion.csv", index=False)

print("\n=== RESUMEN DE LA LISTA DE INSCRITOS ===")
print(f"Total registros inscritos: {len(df_inscritos)}")
print(f"Total dorsales únicos: {df_inscritos['dorsal'].nunique()}")
duplicados_inscritos = df_inscritos.duplicated(subset=['dorsal']).sum()
print(f"Dorsales duplicados técnicos: {duplicados_inscritos}")


print("\n=== MUESTRA SEGURA DE RESULTADOS (SIN NOMBRES NI DORSALES) ===")
cols_seguras_carrera = ["id_carrera", "pagina_origen", "puesto", "puesto_sexo", "puesto_categoria", "tiempo_oficial"]
display(todos_los_resultados["carrera_01"][cols_seguras_carrera].head(5))

print("\n=== MUESTRA SEGURA DE INSCRITOS (SIN NOMBRES NI DORSALES) ===")
cols_seguras_inscritos = ["pagina_origen", "pagado", "estado_inscripcion"]
display(df_inscritos[cols_seguras_inscritos].head(5))

=== REGISTRO DE AUDITORÍA Y COBERTURA (CARRERAS) ===


,id_carrera,evento_id,paginas_totales,total_participantes,duplicados_dorsal,estado
0,carrera_01,3210,60,1489,0,OK
1,carrera_02,3211,38,929,0,OK
2,carrera_03,3212,32,795,0,OK
3,carrera_04,3213,35,875,0,OK
4,carrera_05,3214,27,659,0,OK
5,carrera_06,3215,26,643,0,OK



=== RESUMEN DE LA LISTA DE INSCRITOS ===
Total registros inscritos: 2345
Total dorsales únicos: 2343
Dorsales duplicados técnicos: 2

=== MUESTRA SEGURA DE RESULTADOS (SIN NOMBRES NI DORSALES) ===


,id_carrera,pagina_origen,puesto,puesto_sexo,puesto_categoria,tiempo_oficial
0,carrera_01,1,1,M-1,SNRM-1,00:20:30
1,carrera_01,1,2,M-2,SNRM-2,00:21:00
2,carrera_01,1,3,M-3,SUB23M-1,00:21:01
3,carrera_01,1,4,M-4,SNRM-3,00:21:19
4,carrera_01,1,5,M-5,MS1M-1,00:21:26



=== MUESTRA SEGURA DE INSCRITOS (SIN NOMBRES NI DORSALES) ===


,pagina_origen,pagado,estado_inscripcion
0,1,"27-02-2026, 11:46",
1,1,"27-02-2026, 11:49",
2,1,"27-02-2026, 11:53",
3,1,"27-02-2026, 12:01",
4,1,"27-02-2026, 12:03",
